# Approach: Model 6 - Complete collaboration with addition of:
- HT-MIA
- Min-K%++
- Extended K values
- Token-level LLR
- Soft Voting Ensemble

In [ ]:
#2
import sys
sys.path.insert(0, '/content/drive/MyDrive/model6')

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/model6/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#3
import os
import pickle
import warnings
warnings.filterwarnings('ignore')
import torch
from models import load_data, load_models_and_tokenizer
from batches import process_dataset_in_batches, save_features, load_features
from build_membership_classifier import build_membership_classifier, compute_zlib_baseline, generate_submission

os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# 4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

splits = load_data()

Using device: cuda
Loading dataset...
Train:      50000 samples  (25000 members)
Validation: 10000 samples  (5000 members)
Test:       15000 samples


In [ ]:
#5 - load models
# Skip this cell if loading from cached features.
tokenizer, model_135m, model_360m, ref_model_135m, ref_model_360m = load_models_and_tokenizer(device)

Using model dtype: torch.bfloat16


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading models...


...

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Models loaded: 135M, 360M, and their base reference models.


In [ ]:
#6 - train
train_texts  = splits["train"]["text"].tolist()
train_labels = splits["train"]["is_member"].tolist()

train_135m = process_dataset_in_batches(
    model_135m, tokenizer, train_texts, train_labels,
    "135M", batch_size=500, ref_model=ref_model_135m
)
train_360m = process_dataset_in_batches(
    model_360m, tokenizer, train_texts, train_labels,
    "360M", batch_size=500, ref_model=ref_model_360m
)

train_135m.to_csv(f"{SAVE_DIR}train_135m.csv", index=False)
train_360m.to_csv(f"{SAVE_DIR}train_360m.csv", index=False)
print("Train features saved.")

Processing 50000 samples in batches of 500...
Batch 1/100: 500 samples



Analyzing with 135M: 100%|██████████| 500/500 [00:38<00:00, 12.83it/s]


Batch 2/100: 500 samples



Analyzing with 135M: 100%|██████████| 500/500 [00:37<00:00, 13.26it/s]


Batch 3/100: 500 samples


...

Batch 99/100: 500 samples



Analyzing with 360M: 100%|██████████| 500/500 [00:40<00:00, 12.22it/s]


Batch 100/100: 500 samples



Analyzing with 360M: 100%|██████████| 500/500 [00:40<00:00, 12.23it/s]


Completed processing 50000 samples with 360M
Train features saved.


In [ ]:
# checking new features added
print(train_135m.columns.tolist())
print(f"Number of features: {len(train_135m.columns)}")

['perplexity', 'loss', 'mean_token_prob', 'min_token_prob', 'max_token_prob', 'std_token_prob', 'mean_entropy', 'top_k_mass', 'low_conf_ratio', 'min_k10_prob', 'min_k20_prob', 'min_k30_prob', 'min_k10_pp', 'min_k20_pp', 'min_k30_pp', 'num_tokens', 'ht_mia_score_k10', 'ht_mia_score_k20', 'ht_mia_score_k30', 'ht_mia_score_k40', 'ht_mia_score_k50', 'llr_mean', 'llr_std', 'llr_min', 'llr_max', 'llr_k10', 'llr_k30', 'label', 'text_length']
Number of features: 29


In [ ]:
#6 - val
val_texts  = splits["validation"]["text"].tolist()
val_labels = splits["validation"]["is_member"].tolist()

val_135m = process_dataset_in_batches(
    model_135m, tokenizer, val_texts, val_labels,
    "135M", batch_size=500, ref_model=ref_model_135m
)
val_360m = process_dataset_in_batches(
    model_360m, tokenizer, val_texts, val_labels,
    "360M", batch_size=500, ref_model=ref_model_360m
)

val_135m.to_csv(f"{SAVE_DIR}val_135m.csv", index=False)
val_360m.to_csv(f"{SAVE_DIR}val_360m.csv", index=False)
print("Validation features saved.")

Processing 10000 samples in batches of 500...
Batch 1/20: 500 samples



Analyzing with 135M: 100%|██████████| 500/500 [00:37<00:00, 13.19it/s]


Batch 2/20: 500 samples



Analyzing with 135M: 100%|██████████| 500/500 [00:37<00:00, 13.26it/s]


Batch 3/20: 500 samples


...

Batch 19/20: 500 samples



Analyzing with 360M: 100%|██████████| 500/500 [00:40<00:00, 12.37it/s]


Batch 20/20: 500 samples



Analyzing with 360M: 100%|██████████| 500/500 [00:40<00:00, 12.43it/s]


Completed processing 10000 samples with 360M
Validation features saved ✅


In [ ]:
#6 - test
test_texts  = splits["test"]["text"].tolist()
test_labels = [0] * len(test_texts)

test_135m = process_dataset_in_batches(
    model_135m, tokenizer, test_texts, test_labels,
    "135M", batch_size=500, ref_model=ref_model_135m
)
test_360m = process_dataset_in_batches(
    model_360m, tokenizer, test_texts, test_labels,
    "360M", batch_size=500, ref_model=ref_model_360m
)

test_135m.to_csv(f"{SAVE_DIR}test_135m.csv", index=False)
test_360m.to_csv(f"{SAVE_DIR}test_360m.csv", index=False)
print("Test features saved.")

Processing 15000 samples in batches of 500...
Batch 1/30: 500 samples



Analyzing with 135M: 100%|██████████| 500/500 [00:37<00:00, 13.20it/s]


Batch 2/30: 500 samples



Analyzing with 135M: 100%|██████████| 500/500 [00:37<00:00, 13.25it/s]


Batch 3/30: 500 samples


...

Batch 29/30: 500 samples



Analyzing with 360M: 100%|██████████| 500/500 [00:40<00:00, 12.32it/s]


Batch 30/30: 500 samples



Analyzing with 360M: 100%|██████████| 500/500 [00:40<00:00, 12.32it/s]


Completed processing 15000 samples with 360M
Test features saved.


In [ ]:
#6 - compile features
features = {
    "train_135m": train_135m,
    "train_360m": train_360m,
    "val_135m":   val_135m,
    "val_360m":   val_360m,
    "test_135m":  test_135m,
    "test_360m":  test_360m,
}

In [ ]:
# Run this cell instead of the extraction cells when resuming a session with saved features.
# import pandas as pd

features = {
    "train_135m": pd.read_csv(f"{SAVE_DIR}train_135m.csv"),
    "train_360m": pd.read_csv(f"{SAVE_DIR}train_360m.csv"),
    "val_135m":   pd.read_csv(f"{SAVE_DIR}val_135m.csv"),
    "val_360m":   pd.read_csv(f"{SAVE_DIR}val_360m.csv"),
    "test_135m":  pd.read_csv(f"{SAVE_DIR}test_135m.csv"),
    "test_360m":  pd.read_csv(f"{SAVE_DIR}test_360m.csv"),
}
print("Features loaded from Drive!")

Features loaded from Drive!


In [ ]:
#8
results = build_membership_classifier(
    features["train_135m"], features["train_360m"],
    features["val_135m"],   features["val_360m"],
)

Building membership inference classifier...
Features: 71

Training set:   50000 samples
Validation set: 10000 samples

Training Logistic Regression...
  AUC:           0.9336
  TPR@FPR=0.1:   0.8272  (actual FPR: 0.1002)
  Accuracy:      0.8810

Training Random Forest...
  AUC:           0.9399
  TPR@FPR=0.1:   0.8284  (actual FPR: 0.1016)
  Accuracy:      0.8717

  Top 10 Most Important Features (Random Forest):
    llr_k10_360m: 0.1133
    llr_k30_360m: 0.0717
    ht_mia_score_k40_360m: 0.0627
    loss_ratio: 0.0587
    ht_mia_score_k30_360m: 0.0520
    ht_mia_score_k20_360m: 0.0494
    loss_diff: 0.0335
    perplexity_ratio: 0.0322
    llr_mean_360m: 0.0286
    ht_mia_score_k50_360m: 0.0253

Training XGBoost...
  AUC:           0.9470
  TPR@FPR=0.1:   0.8302  (actual FPR: 0.1000)
  Accuracy:      0.8772

  Top 10 Most Important Features (XGBoost):
    llr_k10_360m: 0.1980
    llr_k30_360m: 0.1270
    ht_mia_score_k30_360m: 0.0732
    loss_ratio: 0.0627
    loss_diff: 0.0483
    perp

In [ ]:
#9
# had to insert this at runtime because something went wrong with the import of zlib

import zlib
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

def compute_zlib_baseline(val_df):
    scores = []
    for text in val_df["text"]:
        encoded = text.encode("utf-8", errors="replace")
        ratio = len(zlib.compress(encoded)) / max(len(encoded), 1)
        scores.append(-ratio)  # negate: lower compression = more likely member

    scores = np.array(scores)
    labels = val_df["is_member"].values

    auc = roc_auc_score(labels, scores)
    fpr, tpr, _ = roc_curve(labels, scores)
    idx = np.argmin(np.abs(fpr - 0.1))
    tpr_at_fpr_01 = tpr[idx]

    print(f"\nZlib Entropy Baseline:")
    print(f"  AUC:         {auc:.4f}")
    print(f"  TPR@FPR=0.1: {tpr_at_fpr_01:.4f}")

    return {"auc": auc, "tpr_at_fpr_01": tpr_at_fpr_01}


zlib_result = compute_zlib_baseline(splits["validation"])



Zlib Entropy Baseline:
  AUC:         0.5075
  TPR@FPR=0.1: 0.1028


In [ ]:
#10
for name, r in results.items():
    print(f"\n{name}:  AUC={r['auc']:.4f}  TPR@FPR=0.1={r['tpr_at_fpr_01']:.4f}  Acc={r['accuracy']:.4f}")

best_name = max(results, key=lambda x: results[x]["auc"])
best      = results[best_name]
print(f"\nBest: {best_name}  AUC={best['auc']:.4f}  TPR@FPR=0.1={best['tpr_at_fpr_01']:.4f}")

classifier_path = SAVE_DIR + "membership_classifier.pkl"
with open(classifier_path, "wb") as f:
    pickle.dump(best, f)
print(f"Saved classifier to {classifier_path}")

submission_path = "membership_inference_submission.csv"
submission_df = generate_submission(
    features["test_135m"], features["test_360m"],
    best, splits["test"],
    output_path=submission_path,
)
print(submission_df.head())


Logistic Regression:  AUC=0.9336  TPR@FPR=0.1=0.8272  Acc=0.8810

Random Forest:  AUC=0.9399  TPR@FPR=0.1=0.8284  Acc=0.8717

XGBoost:  AUC=0.9470  TPR@FPR=0.1=0.8302  Acc=0.8772

Gradient Boosting:  AUC=0.9447  TPR@FPR=0.1=0.8232  Acc=0.8770

Ensemble:  AUC=0.9487  TPR@FPR=0.1=0.8310  Acc=0.8786

Best: Ensemble  AUC=0.9487  TPR@FPR=0.1=0.8310
Saved classifier to /content/drive/MyDrive/model6/membership_classifier.pkl
Preparing test features for prediction...
Preparing test features for prediction...
Generating predictions on test set...
Preparing test features for prediction...
Generating predictions on test set...
Preparing test features for prediction...
Generating predictions on test set...
Preparing test features for prediction...
Generating predictions on test set...
Submission saved: membership_inference_submission.csv  (15000 rows)
Prediction range: [0.0014, 0.9861]
       id  is_member
0   32656   0.970461
1  137254   0.050147
2  134884   0.522761
3   33065   0.656419
4   343

In [ ]:
#11 download
from google.colab import files
files.download('membership_inference_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>